In [1]:
import pandas as pd

# define important cols
keep_cols = ['buyer_state', 'drug_name', 'transaction_date', 'calc_base_wt_in_gm']

chunks = pd.read_csv(
    "../../data/DEA_2.csv.gz", 
    usecols=keep_cols, 
    chunksize=1000000, 
    compression='gzip',
    low_memory=False
)

aggregated_results = []

print("Starting extraction...")
for i, chunk in enumerate(chunks):
    # 1. CLEAN DRUG NAMES FIRST (Strip spaces and uppercase)
    chunk['drug_name'] = chunk['drug_name'].astype(str).str.strip().str.upper()
    
    # 2. UPDATED MASK
    targets = ['OXYCODONE', 'HYDROCODONE', 'FENTANYL', 'FENTANYL BASE']
    mask = chunk['drug_name'].isin(targets)
    small_chunk = chunk[mask].copy()
    
    if not small_chunk.empty:
        # Normalize name to remove "Base"
        small_chunk['drug_name'] = small_chunk['drug_name'].replace({'FENTANYL BASE': 'FENTANYL'})
        
        # Extract year safely (if date is MMDDYYYY)
        # Using string slicing is MUCH faster and less crash-prone than pd.to_datetime
        small_chunk['year'] = small_chunk['transaction_date'].astype(str).str[-4:].astype(int)
        
        # Aggregate
        chunk_summed = small_chunk.groupby(['buyer_state', 'year', 'drug_name'])['calc_base_wt_in_gm'].sum().reset_index()
        aggregated_results.append(chunk_summed)
    
    if i % 10 == 0:
        print(f"Chunk {i}: Found {len(small_chunk)} relevant rows.")

# Check before concat
if not aggregated_results:
    print("❌ ERROR: No data found! Check drug names in raw file.")
else:
    df_mini = pd.concat(aggregated_results)
    df_final_sum = df_mini.groupby(['buyer_state', 'year', 'drug_name'])['calc_base_wt_in_gm'].sum().reset_index()
    print(f"✅ SUCCESS: Found {len(df_final_sum)} state/year/drug combinations.")
    df_final_sum.to_csv('../../data/dea_summary_pre_aggregated.csv', index=False)

Starting extraction...
Chunk 0: Found 767817 relevant rows.
Chunk 10: Found 750388 relevant rows.
Chunk 20: Found 747667 relevant rows.
Chunk 30: Found 738138 relevant rows.
Chunk 40: Found 751581 relevant rows.
Chunk 50: Found 755490 relevant rows.
Chunk 60: Found 745853 relevant rows.
Chunk 70: Found 748671 relevant rows.
Chunk 80: Found 730105 relevant rows.
Chunk 90: Found 740931 relevant rows.
Chunk 100: Found 741717 relevant rows.
Chunk 110: Found 744770 relevant rows.
Chunk 120: Found 745256 relevant rows.
Chunk 130: Found 731698 relevant rows.
Chunk 140: Found 734544 relevant rows.
Chunk 150: Found 731050 relevant rows.
Chunk 160: Found 719804 relevant rows.
Chunk 170: Found 711489 relevant rows.
Chunk 180: Found 722606 relevant rows.
Chunk 190: Found 727014 relevant rows.
Chunk 200: Found 724074 relevant rows.
Chunk 210: Found 727080 relevant rows.
Chunk 220: Found 717122 relevant rows.
Chunk 230: Found 724916 relevant rows.
Chunk 240: Found 717173 relevant rows.
Chunk 250: Fo

In [2]:
# inspect the mini file
df = pd.read_csv('../../data/dea_summary_pre_aggregated.csv')

print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1683 entries, 0 to 1682
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   buyer_state         1683 non-null   object 
 1   year                1683 non-null   int64  
 2   drug_name           1683 non-null   object 
 3   calc_base_wt_in_gm  1683 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 52.7+ KB
None


,buyer_state,year,drug_name,calc_base_wt_in_gm
0,AK,2006,FENTANYL,1564.526023
1,AK,2006,HYDROCODONE,73936.970520
2,AK,2006,OXYCODONE,132344.307041
3,AK,2007,FENTANYL,1575.875489
4,AK,2007,HYDROCODONE,80968.424733


In [3]:
# Clean, get to format that can merge with NCHS

print("Pivoting drug columns...")
df_pivot = df.pivot_table(
    index=['buyer_state', 'year'], 
    columns='drug_name', 
    values='calc_base_wt_in_gm'
).reset_index()

# Clean col names
df_pivot.columns.name = None 
df_pivot = df_pivot.rename(columns={
    'buyer_state': 'state',
    'HYDROCODONE': 'hydro_gms',
    'OXYCODONE': 'oxy_gms',
    'FENTANYL': 'fent_gms'
})

# Fill NAs and save
df_pivot = df_pivot.fillna(0)
df_pivot.to_csv('../../data/dea_final.csv', index=False)
print("Transformation complete! No more crashes.")

Pivoting drug columns...
Transformation complete! No more crashes.


In [4]:
import numpy as np
# Data from 2000, 2005
# state mapping dict
state_map = {
    'ALABAMA': 'AL', 'ALASKA': 'AK', 'ARIZONA': 'AZ', 'ARKANSAS': 'AR', 'CALIFORNIA': 'CA',
    'COLORADO': 'CO', 'CONNECTICUT': 'CT', 'DELAWARE': 'DE', 'FLORIDA': 'FL', 'GEORGIA': 'GA',
    'HAWAII': 'HI', 'IDAHO': 'ID', 'ILLINOIS': 'IL', 'INDIANA': 'IN', 'IOWA': 'IA',
    'KANSAS': 'KS', 'KENTUCKY': 'KY', 'LOUISIANA': 'LA', 'MAINE': 'ME', 'MARYLAND': 'MD',
    'MASSACHUSETTS': 'MA', 'MICHIGAN': 'MI', 'MINNESOTA': 'MN', 'MISSISSIPPI': 'MS', 'MISSOURI': 'MO',
    'MONTANA': 'MT', 'NEBRASKA': 'NE', 'NEVADA': 'NV', 'NEW HAMPSHIRE': 'NH', 'NEW JERSEY': 'NJ',
    'NEW MEXICO': 'NM', 'NEW YORK': 'NY', 'NORTH CAROLINA': 'NC', 'NORTH DAKOTA': 'ND', 'OHIO': 'OH',
    'OKLAHOMA': 'OK', 'OREGON': 'OR', 'PENNSYLVANIA': 'PA', 'RHODE ISLAND': 'RI', 'SOUTH CAROLINA': 'SC',
    'SOUTH DAKOTA': 'SD', 'TENNESSEE': 'TN', 'TEXAS': 'TX', 'UTAH': 'UT', 'VERMONT': 'VT',
    'VIRGINIA': 'VA', 'WASHINGTON': 'WA', 'WEST VIRGINIA': 'WV', 'WISCONSIN': 'WI', 'WYOMING': 'WY',
    'DISTRICT OF COLUMBIA': 'DC'
}

# load and clean 2000, 2005 data
df_manual = pd.read_excel("../../data/DEA_2000_2005.xlsx")
# df_manual = df_manual.rename(columns={"Year":'year', "State":'state'})
df_manual['state'] = df_manual['state'].str.upper().str.strip().map(state_map)
df_manual = df_manual[['state', 'year', 'hydro_gms', 'oxy_gms', 'fent_gms']]

df_manual.head()

,state,year,hydro_gms,oxy_gms,fent_gms
0,AK,2000,27018.40,74395.72,NaN
1,WV,2000,152541.08,197056.02,NaN
2,DE,2000,22179.56,79671.76,NaN
3,FL,2000,983039.12,1569180.46,NaN
4,CT,2000,99110.22,338763.08,NaN


In [5]:
df_manual.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   state      150 non-null    object 
 1   year       150 non-null    int64  
 2   hydro_gms  150 non-null    float64
 3   oxy_gms    150 non-null    float64
 4   fent_gms   100 non-null    float64
dtypes: float64(3), int64(1), object(1)
memory usage: 6.0+ KB


In [6]:
df_2006_16 = pd.read_csv('../../data/dea_final.csv')
df_2006_16 = df_2006_16.loc[df_2006_16.state!="DC"].reset_index(drop=True)
df_2006_16.head()

,state,year,fent_gms,hydro_gms,oxy_gms
0,AK,2006,1564.526023,73936.970520,132344.307041
1,AK,2007,1575.875489,80968.424733,141050.065458
2,AK,2008,1706.778672,86069.808517,153192.816393
3,AK,2009,1646.588400,89413.086368,168255.904702
4,AK,2010,1852.650595,90779.457730,174587.019594


In [7]:
df_manual.head()

,state,year,hydro_gms,oxy_gms,fent_gms
0,AK,2000,27018.40,74395.72,NaN
1,WV,2000,152541.08,197056.02,NaN
2,DE,2000,22179.56,79671.76,NaN
3,FL,2000,983039.12,1569180.46,NaN
4,CT,2000,99110.22,338763.08,NaN


In [8]:
dea_master = pd.concat([df_manual, df_2006_16], ignore_index=True)
dea_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   state      700 non-null    object 
 1   year       700 non-null    int64  
 2   hydro_gms  700 non-null    float64
 3   oxy_gms    700 non-null    float64
 4   fent_gms   650 non-null    float64
dtypes: float64(3), int64(1), object(1)
memory usage: 27.5+ KB


In [9]:
# initialize grid with all state-year combinations
states = dea_master['state'].unique()
years = range(2000, 2017)
full_grid = pd.MultiIndex.from_product([states, years], names=['state', 'year']).to_frame(index=False)

# add data
dea_full = pd.merge(full_grid, dea_master, on=['state', 'year'], how='left')

# sort and fill drug gms, interpolate missing years
drug_cols = ['hydro_gms', 'oxy_gms', 'fent_gms']
dea_full[drug_cols] = dea_full.groupby('state')[drug_cols].transform(lambda x: x.interpolate(method='linear'))

# Backfill Fentanyl for 2000 (since your data starts in 2001)
dea_full['fent_gms'] = dea_full.groupby('state')['fent_gms'].transform(lambda x: x.bfill())

# inspect result
print(dea_full[dea_full['state'] == 'AL'].head(7))

    state  year     hydro_gms       oxy_gms      fent_gms
340    AL  2000  4.742733e+05  2.990482e+05   2793.510000
341    AL  2001  5.034673e+05  3.456079e+05   2793.510000
342    AL  2002  5.771365e+05  3.688052e+05   3737.692500
343    AL  2003  6.508058e+05  3.920025e+05   4681.875000
344    AL  2004  7.244751e+05  4.151999e+05   5626.057500
345    AL  2005  7.981443e+05  4.383972e+05   6570.240000
346    AL  2006  6.506259e+07  1.538785e+06  15680.581498


In [10]:
dea_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 850 entries, 0 to 849
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   state      850 non-null    object 
 1   year       850 non-null    int64  
 2   hydro_gms  850 non-null    float64
 3   oxy_gms    850 non-null    float64
 4   fent_gms   850 non-null    float64
dtypes: float64(3), int64(1), object(1)
memory usage: 33.3+ KB


In [11]:
# Export the final file
dea_full.to_csv('../../data/dea_full_interpolated.csv', index=False)